In [1]:
!pip install --upgrade transformers -q

In [2]:
import pandas as pd
columns = ["qid",  "docid", "rank", "score", "querytext", "doctext"]

pairs = pd.read_csv(
    "/kaggle/input/datasets/hatemamine/pairofqueryand10releventtextmrtydi/3cross_endocder_rrf.txt",
    sep="\t",
    header=None,
   names=columns
)
pairs=pairs[["qid", "docid", "rank"]]
pairs.head()
pairs=pairs[pairs["rank"] < 11]
qid_to_docs = pairs.groupby("qid")["docid"].apply(list).to_dict()
#qid_to_docs[15513]

In [3]:
from huggingface_hub import hf_hub_download
import torch
organization_dataset_id= "hatemestinbejaia/ExperimentDATA_knowledge_distillation_vs_fine_tuning"
collection_name= "mrtydi-collection-arabic-preprocess-for-araelectra"
queries_name= "mrtydi-queries-arabic-preprocess-for-araelectra"
qerl_file ="qrels.test.mrtydi.txt"

hf_hub_download(repo_id=organization_dataset_id, filename=qerl_file, local_dir="./", repo_type="dataset")

from datasets import load_dataset
datasetC = load_dataset(organization_dataset_id, collection_name, trust_remote_code=True)
print(datasetC['train'][100])
print(datasetC)

from datasets import load_dataset
datasetQ = load_dataset(organization_dataset_id, queries_name, trust_remote_code=True)
datasetQ = datasetQ["train"]
print(datasetQ)
print(datasetQ[0])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'hatemestinbejaia/ExperimentDATA_knowledge_distillation_vs_fine_tuning' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'hatemestinbejaia/ExperimentDATA_knowledge_distillation_vs_fine_tuning' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


{'docid': '38#2', 'title': 'رياضيات', 'text': 'و لقد نشأ علم الرياضيات عندما قاس الإنسان ما شاهده من ظواهر طبيعية و بناء على فطرة وخاصية فيه ألا و هي اهتمامه بقياس كل ما حوله إلى جانب احتياجاته العملية فهكذا كانت هناك ضرورة لقياس قسمة الأقوات ( الطعام ) بين أفراد العائلة و قياس الوقت و الفصول و المحاصيل الزراعية و تقسيم الأراضي و غنائم الحملات الحربية و المحاسبة للتمكن من الإتجار إلى جانب علم الملاحة حيث الاهتداء بالنجوم في السفر و الترحال للتجارة و السياحة والقياسات اللازمة لتشييد الأبنية و المدن .'}
DatasetDict({
    train: Dataset({
        features: ['docid', 'title', 'text'],
        num_rows: 2106586
    })
})
Dataset({
    features: ['id', 'text'],
    num_rows: 1081
})
{'id': 15513, 'text': 'كم عدد مرات فوز الأوروغواي ببطولة كاس العالم لكرو القدم ؟'}


In [4]:
# Extract columns as lists
docids = datasetC['train']['docid']
texts = datasetC['train']['text']

# Build dictionary in one line
corpus = dict(zip(docids, texts))

In [5]:
import tqdm 
queries = {}
for i in tqdm.trange(len(datasetQ)):
    queries[datasetQ['id'][i]]= datasetQ['text'][i]
    #break

100%|██████████| 1081/1081 [00:00<00:00, 2143.73it/s]


In [6]:
from transformers import AutoProcessor, AutoModelForCausalLM

MODEL_ID = "google/gemma-4-E4B-it"

# Load model
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto"
)

model.config.use_cache = False

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

In [8]:
import torch
import re
import time
from tqdm import tqdm
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
results = {}

#MAX_DOCS = 10          # limit candidates per query (important for GPU)
#MAX_DOC_LEN = 1000     # truncate long documents
MAX_NEW_TOKENS = 16    # reduce generation cost

for qid, docids in tqdm(qid_to_docs.items(), total=len(qid_to_docs), desc="Processing queries"):

    query = queries[qid]
    t0 = time.perf_counter()

    # ---- Build candidate documents (LIMITED) ----
    docs_text = []
    docs_id = []

    for i, docid in enumerate(docids, start=0):
        doc = corpus[docid]
        docs_text.append(f"الوثيقة {i}:\n{doc}\n")
        docs_id.append(docid)

    docs_block = "\n".join(docs_text)

    # ---- Prompt ----
    messages = [
        {
            "role": "system",
            "content": "أنت نظام لترتيب المستندات حسب الصلة. مهمتك اختيار رقم الوثيقة الأكثر صلة فقط. يجب أن يكون الإخراج رقمًا صحيحًا واحدًا فقط بدون أي نص إضافي، بدون كلمات، بدون أقواس، بدون علامات. مثال: 5"
        },
        {
            "role": "user",
            "content": f"""السؤال:
{query}

قائمة الوثائق:
{docs_block}

ANSWER: <رقم الوثيقة>"""
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(text=text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    # ---- GPU-safe inference ----
    with torch.inference_mode(), torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )

    response = processor.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    )

    # ---- Cleanup GPU memory ----
    del inputs
    del outputs
    torch.cuda.empty_cache()

    # ---- Parse response ----
    ans = processor.parse_response(response)

    if isinstance(ans, dict) and "content" in ans:
        ans_text = ans["content"]
    elif isinstance(ans, str):
        ans_text = ans
    else:
        print("⚠ Unrecognized format:", ans)
        continue

    # ---- FIXED regex ----
    ans_text = response.strip()
    #print(ans_text)
    #print(type(ans_text))
    try:
        best_rank = int(ans_text)
    except ValueError:
        print("Invalid output (not an integer):", ans_text)
        continue
    if -1 <best_rank< 10:
            best_docid = docs_id[best_rank]
    else:
            print("⚠ Rank out of range:", best_rank)
            continue

    # ---- Save output ----
    with open("gemma_run.txt", "a", encoding="utf-8") as f:
        f.write(f"{qid} Q0 {best_docid} 1\n")

    t1 = time.perf_counter()
    del response

    torch.cuda.empty_cache()

Processing queries:  21%|██▏       | 230/1081 [25:01<1:32:35,  6.53s/it]


OutOfMemoryError: CUDA out of memory. Tried to allocate 504.00 MiB. GPU 1 has a total capacity of 14.56 GiB of which 299.81 MiB is free. Including non-PyTorch memory, this process has 14.27 GiB memory in use. Of the allocated memory 13.64 GiB is allocated by PyTorch, and 510.14 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)